In [13]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
# 1. Load the cleaned data
DATA_PATH = '../Data/processed/clean_products.csv'
df = pd.read_csv(DATA_PATH)

print(f" Data loaded successfully. Total products: {len(df)}")

 Data loaded successfully. Total products: 1666


 TF-IDF Vectorisation:

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text

In [15]:
# 1. Define custom stop words
# We add vendor names here so the AI doesn't recommend two products 
# just because they are both from 'Telemart' or 'PriceOye'.
vendor_stop_words = ['mega', 'pk', 'telemart', 'priceoye', 'paklap', 'computerzone']
final_stop_words = list(text.ENGLISH_STOP_WORDS.union(vendor_stop_words))

In [16]:
# 2. Initialize the Vectorizer
# ngram_range=(1, 2) creates both unigrams (single words) and bigrams (pairs)
tfidf = TfidfVectorizer(
    stop_words=final_stop_words, 
    ngram_range=(1, 2), # Captures "gaming laptop", "core i7", "12gb ram"
    min_df=2            # Ignore words that appear in only 1 product (removes noise)
)

In [17]:
# 3. Create the Matrix
# This turns every product's 'clean_content' into a vector of numbers
tfidf_matrix = tfidf.fit_transform(df['clean_content'].fillna(''))

print(f"TF-IDF Matrix created with shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.get_feature_names_out())} unique tokens.")

TF-IDF Matrix created with shape: (1666, 2209)
Vocabulary size: 2209 unique tokens.


Search Engine:

In [18]:
def search(query, category=None, min_price=None, max_price=None, top_n=10):
    # 1. Transform query and calculate similarity scores
    query_vec = tfidf.transform([query.lower()])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # 2. Create a Boolean Mask for Filtering
    # We start with all True and 'AND' each filter
    mask = np.ones(len(df), dtype=bool)
    
    if category:
        mask &= (df['category'].str.lower() == category.lower())
    
    if min_price is not None:
        mask &= (df['discounted_price'] >= min_price)
        
    if max_price is not None:
        mask &= (df['discounted_price'] <= max_price)
    
    # 3. Apply the filter mask to the dataset and similarity scores
    filtered_indices = np.where(mask)[0]
    filtered_sims = sim_scores[filtered_indices]
    
    # 4. Extract Top Results
    # Sort the subset indices by their similarity scores in descending order
    top_subset_indices = filtered_sims.argsort()[::-1][:top_n]
    final_indices = filtered_indices[top_subset_indices]
    
    # Construct the results DataFrame
    results = df.iloc[final_indices].copy()
    results['relevance_score'] = filtered_sims[top_subset_indices]
    
    return results[['id', 'vendor', 'brand', 'title', 'category', 'discounted_price', 'relevance_score']]

In [19]:
# Query: Samsung 8GB RAM
search("Samsung 8GB RAM", top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
35,35,MEGA.PK,Samsung,Samsung Galaxy S23 8GB RAM 256GB Storage Non PTA,Mobile,222999.0,0.463896
67,67,MEGA.PK,Samsung,Samsung Galaxy Z Flip 4 8GB RAM 512GB Storage,Mobile,309999.0,0.457028
170,170,MEGA.PK,Samsung,Samsung Galaxy Z Flip 4 8GB RAM 256GB Storage ...,Mobile,0.0,0.454711
34,34,MEGA.PK,Samsung,Samsung Galaxy S23 Plus 8GB RAM 256GB Storage ...,Mobile,252999.0,0.454028
132,132,MEGA.PK,Samsung,Samsung Galaxy A33 8GB RAM 128GB Storage 5G PT...,Mobile,111999.0,0.445496


In [20]:
# Query: iPhone 15 Pro Max
search("iPhone 15 Pro Max", top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
4,4,MEGA.PK,Apple,Apple Iphone 15 Pro Max,Mobile,0.0,0.687878
113,113,MEGA.PK,Apple,Apple iPhone 14 Pro Max,Mobile,419999.0,0.448249
121,121,MEGA.PK,Apple,Apple iPhone 14 Pro Max,Mobile,334999.0,0.448249
1188,1188,PriceOye,Pro,i8 Pro Max Smart Watch (44mm),Watch,1999.0,0.426421
1456,1456,PriceOye,Apple,Apple iPhone 14 Pro Max,Mobile,669999.0,0.411262


In [21]:
# Query: Gaming Laptop
search("Gaming Laptop", top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
724,724,Paklap,Hp,HP Victus 15 Gaming Laptop,Laptop,200900.0,0.457450
419,419,ComputerZone,HP,Hp Victus Gaming Laptop 15-FA1040NE,Laptop,328900.0,0.453704
404,404,ComputerZone,HP,HP Victus 15-FA0025NR Gaming Laptop,Laptop,245000.0,0.406222
408,408,ComputerZone,HP,HP Victus 15-FA1093DX Gaming Laptop,Laptop,263900.0,0.406222
407,407,ComputerZone,HP,HP Victus 15-FB0028NR Gaming Laptop,Laptop,258900.0,0.397932


In [22]:
# Query: Wireless Noise Cancelling Headphones
search("Wireless Noise Cancelling Headphones", top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
897,897,PriceOye,Sony,Sony Wf-1000XM4 Wireless Noise Cancelling Hea...,Earbuds,46999.0,0.688707
949,949,PriceOye,Sony,Sony Wireless Noise Cancelling Headphones WH-1...,Earbuds,59999.0,0.642495
811,811,PriceOye,Joyroom,Joyroom Upgraded Noise Cancelling Earbuds (TA2),Earbuds,8399.0,0.398261
1061,1061,PriceOye,Tronsmart,Tronsmart Apollo Hybrid Active Noise Cancellin...,Earbuds,9599.0,0.323148
963,963,PriceOye,Faster,Faster True Wireless Noise Reduction Earbuds (...,Earbuds,3999.0,0.295506


In [23]:
# Query: "Apple" strictly in the "Watch" category
search("Apple", category="Watch", top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
1190,1190,PriceOye,Apple,Apple Logo Watch 8 Ultra,Watch,6399.0,0.429338
1268,1268,PriceOye,Apple,Apple Watch Series 7 (45mm),Watch,90999.0,0.268316
1175,1175,PriceOye,Apple,Apple Watch Series 8 (45mm),Watch,118999.0,0.259361
1197,1197,PriceOye,Apple,Apple Watch Series 8 (41mm),Watch,114999.0,0.256933
1340,1340,PriceOye,T200,T200 Plus Smart Watch,Watch,3799.0,0.000000


In [24]:
# Query: "Smartwatch" under 15,000 PKR
search("Smartwatch", max_price=15000, top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
1246,1246,PriceOye,KD99,KD99 Bluetooth Calling Smartwatch,Watch,3599.0,0.477901
1298,1298,PriceOye,TG,TG-38 Ultra Smartwatch,Watch,6199.0,0.470213
1293,1293,PriceOye,AUKEY,AUKEY Smartwatch LS-02,Watch,5299.0,0.373322
1153,1153,PriceOye,G,G-Tide S1 Lite Bluetooth Calling Smartwatch,Watch,8199.0,0.340293
1256,1256,PriceOye,Haino,Haino Teko T85 Max SmartWatch,Watch,6699.0,0.332873


In [25]:
# Query: Sony
search("Sony", top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
990,990,PriceOye,Sony,Sony Truly Wireless Headphones WF-C500,Earbuds,20999.0,0.626874
1058,1058,PriceOye,Sony,Sony Headphones WI-XB400,Earbuds,15999.0,0.618103
1097,1097,PriceOye,Sony,Sony Headphones WI-C310,Earbuds,6999.0,0.618103
1096,1096,PriceOye,Sony,Sony Headphones WI-C200,Earbuds,5999.0,0.618103
933,933,PriceOye,Sony,Sony Headphones WI-SP510,Earbuds,14999.0,0.618103


In [26]:
# Query: Mechanical Keyboard
search("Mechanical Keyboard", min_price=5000, max_price=15000, top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
1591,1591,PriceOye,Nokia,Nokia 110 4G,Mobile,5999.0,0.0
1580,1580,PriceOye,Jazz,Jazz Digit Shine 4G,Mobile,7599.0,0.0
1570,1570,PriceOye,Nokia,Nokia 105,Mobile,5249.0,0.0
1553,1553,PriceOye,Jazz,Jazz Digit Crown 4G,Mobile,6299.0,0.0
1548,1548,PriceOye,A17,itel A17,Mobile,11799.0,0.0


In [27]:
# Query: HP Laptop between 150,000 and 250,000 PKR
search("HP Laptop", min_price=150000, max_price=250000, top_n=5)

,id,vendor,brand,title,category,discounted_price,relevance_score
772,772,Paklap,Hp,HP ZBook 15 G5 Core i7,Laptop,199900.0,0.542432
379,379,ComputerZone,HP,HP 15S-FQ5295NIA Laptop,Laptop,154900.0,0.487806
389,389,ComputerZone,HP,HP 15S-FQ5297NIA Laptop,Laptop,189500.0,0.484876
392,392,ComputerZone,HP,HP Pavilion 15-EH1114AU Laptop - Power and Per...,Laptop,194500.0,0.482746
394,394,ComputerZone,HP,HP 15-DW4026NIA Laptop,Laptop,197900.0,0.481091


 Recommendation Prototype

In [28]:
# 1. Compute the full similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [29]:
# 2. Define the Recommendation Logic
def recommend(product_id, top_n=5):
    # Find the index of the product that matches the ID
    idx = df[df['id'] == product_id].index[0]
    
    # Get all similarity scores for this product
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort the scores in descending order (highest similarity first)
    # x[1] refers to the similarity score in the (index, score) tuple
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Slice the list: skip index 0 (the product itself) and take top_n
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]
    
    return df.iloc[top_indices][['id', 'vendor', 'brand', 'title', 'discounted_price']]

In [30]:
# 6: VENDOR BIAS AUDIT ---

def run_bias_audit(df, cosine_sim_matrix):
    vendors = df['vendor'].unique()
    audit_results = []

    print(f"--- Running Audit on {len(vendors)} Vendors ---\n")

    for v in vendors:
        # 1. Pick a sample product from this vendor
        sample_product = df[df['vendor'] == v].iloc[0]
        sample_id = sample_product['id']
        
        # 2. Get recommendations (using logic from Section 5)
        idx = df[df['id'] == sample_id].index[0]
        sim_scores = list(enumerate(cosine_sim_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        top_indices = [i[0] for i in sim_scores[1:6]] # Top 5 excluding self
        
        recs = df.iloc[top_indices]
        
        # 3. Analyze diversity
        unique_vendors_count = recs['vendor'].nunique()
        vendor_list = recs['vendor'].unique().tolist()
        
        audit_results.append({
            'Vendor': v,
            'Sample Product': sample_product['title'][:40] + "...",
            'Unique Vendors in Top-5': unique_vendors_count,
            'Rec Vendors': vendor_list
        })

    # 4. Display Results
    bias_df = pd.DataFrame(audit_results)
    avg_diversity = bias_df['Unique Vendors in Top-5'].mean()
    
    print(bias_df[['Vendor', 'Unique Vendors in Top-5', 'Rec Vendors']])
    print(f"\n Average Vendor Diversity Score: {avg_diversity:.2f} / 5.0")
    
    if avg_diversity < 2.0:
        print(" WARNING: High Vendor Bias detected. Results are too store-locked.")
    else:
        print(" SUCCESS: Model shows healthy cross-vendor discovery.")



In [31]:
# Run the audit
run_bias_audit(df, cosine_sim)

--- Running Audit on 5 Vendors ---

         Vendor  Unique Vendors in Top-5  \
0       MEGA.PK                        1   
1  ComputerZone                        3   
2     TechGlobe                        3   
3        Paklap                        5   
4      PriceOye                        1   

                                         Rec Vendors  
0                                          [MEGA.PK]  
1                 [ComputerZone, TechGlobe, MEGA.PK]  
2                    [Paklap, ComputerZone, MEGA.PK]  
3  [Paklap, MEGA.PK, ComputerZone, TechGlobe, Pri...  
4                                         [PriceOye]  

 Average Vendor Diversity Score: 2.60 / 5.0
 SUCCESS: Model shows healthy cross-vendor discovery.


In [44]:
#------section 7: anti-biased RECOMMENDATION LOGIC ---
def recommend_unbiased(product_id, top_n=5):
    """
    Applies a 25% penalty to same-vendor items (>0.8 sim) 
    and forces a minimum of 2 unique vendors in the top results.
    """
    try:
        # 1. Align index and identify source vendor
        idx = df[df['id'] == product_id].index[0]
        source_vendor = df.iloc[idx]['vendor']
        
        # 2. Extract and penalize similarity scores
        sim_scores = cosine_sim[idx].copy()
        
        # Rule 1: The "Store-Fingerprint" Penalty
        # Shaves off 25% similarity if it's the same store and very high match
        for i in range(len(sim_scores)):
            if i == idx: continue 
            if df.iloc[i]['vendor'] == source_vendor and sim_scores[i] > 0.8:
                sim_scores[i] *= 0.75 
        
        # 3. Rank products
        sorted_indices = sim_scores.argsort()[::-1]
        
        # 4. Enforce Diversity (Rule 2)
        final_indices = []
        found_vendors = {source_vendor}
        
        for i in sorted_indices:
            if i == idx: continue
            if len(final_indices) >= top_n: break
            
            current_vendor = df.iloc[i]['vendor']
            
            # THE FIX: If we are at the last available slot (top_n - 1)
            # and we still only have items from the source vendor,
            # we FORCE the search to find a competitor.
            if len(final_indices) == (top_n - 1) and len(found_vendors) < 2:
                if current_vendor != source_vendor:
                    final_indices.append(i)
                    found_vendors.add(current_vendor)
            else:
                final_indices.append(i)
                found_vendors.add(current_vendor)
        
        return df.iloc[final_indices][['id', 'vendor', 'brand', 'title', 'discounted_price']]

    except IndexError:
        return f"Error: Product ID {product_id} not found."



In [46]:

recommend_unbiased(1111, top_n=5)

,id,vendor,brand,title,discounted_price
1334,1334,PriceOye,Dany,Dany Callfit-5 Smart Watch,8499.0
1139,1139,PriceOye,Dany,Dany Smart Fit 3 Smart Watch,5399.0
1040,1040,PriceOye,Audionic,Audionic Bluetooth Neckband (A600),1999.0
1004,1004,PriceOye,Audionic,Audionic Bluetooth Neckband (A400),1599.0
15,15,MEGA.PK,OPPO,OPPO A17 4GB RAM 64GB Storage PTA Approved,44999.0


Final Validation:

In [33]:
# --- 1. DATA & SIMILARITY RE-LOAD ---
# Ensuring the similarity matrix from Section 5 is available
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [34]:
# --- 2. LOGIC WRAPPERS ---
def recommend_original(product_id, top_n=5):
    """The basic logic before anti-bias rules."""
    idx = df[df['id'] == product_id].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]
    return df.iloc[top_indices]

In [47]:
def recommend_unbiased(product_id, top_n=5):
    """The new logic with 25% penalty and hard diversity rule."""
    idx = df[df['id'] == product_id].index[0]
    source_vendor = df.iloc[idx]['vendor']
    sim_scores = cosine_sim[idx].copy()
    
    # Apply Rule 1: 25% Penalty for high-similarity same-vendor items
    for i in range(len(sim_scores)):
        if i != idx and df.iloc[i]['vendor'] == source_vendor and sim_scores[i] > 0.8:
            sim_scores[i] *= 0.75 
            
    sorted_indices = sim_scores.argsort()[::-1]
    final_indices = []
    found_vendors = {source_vendor}
    
    # Apply Rule 2: Hard Minimum 2 Vendors
    for i in sorted_indices:
        if i == idx: continue
        if len(final_indices) >= top_n: break
        
        curr_v = df.iloc[i]['vendor']
        if len(final_indices) == (top_n - 1) and len(found_vendors) < 2:
            if curr_v != source_vendor:
                final_indices.append(i)
                found_vendors.add(curr_v)
        else:
            final_indices.append(i)
            found_vendors.add(curr_v)
    return df.iloc[final_indices]

In [49]:
# --- 3. SIDE-BY-SIDE COMPARISON (Nothing Phone 1) ---
id_to_test = 0
before = recommend_original(id_to_test)
after = recommend_unbiased(id_to_test)

print("### SECTION 8.1: SIDE-BY-SIDE DEMO (ID: 0) ###")
comparison_table = pd.DataFrame({
    'Rank': range(1, 6),
    'Biased_Vendor': before['vendor'].values,
    'Biased_Title': before['title'].str[:35].values,
    'Unbiased_Vendor': after['vendor'].values,
    'Unbiased_Title': after['title'].str[:35].values
})
print(comparison_table.to_string(index=False))

### SECTION 8.1: SIDE-BY-SIDE DEMO (ID: 0) ###
 Rank Biased_Vendor                        Biased_Title Unbiased_Vendor                      Unbiased_Title
    1       MEGA.PK Samsung Galaxy A73 8GB Ram 256GB St         MEGA.PK Samsung Galaxy A73 8GB Ram 256GB St
    2       MEGA.PK Vivo V27e 8GB RAM 256GB Storage PTA         MEGA.PK Vivo V27e 8GB RAM 256GB Storage PTA
    3       MEGA.PK Samsung Galaxy S23 8GB RAM 256GB St         MEGA.PK Samsung Galaxy S23 8GB RAM 256GB St
    4       MEGA.PK Samsung Galaxy Z Flip 4 8GB RAM 256         MEGA.PK Samsung Galaxy Z Flip 4 8GB RAM 256
    5       MEGA.PK Samsung Galaxy S23 Plus 8GB RAM 256        PriceOye              Samsung Galaxy A52s 5G


In [37]:
# --- 4. CALCULATE MLFLOW METRIC ---
# We sample 100 products to get a statistically significant average
sample_pids = df['id'].sample(100, random_state=42)
diversity_scores = [recommend_unbiased(pid)['vendor'].nunique() for pid in sample_pids]

mlflow_metric = np.mean(diversity_scores)

print("\n" + "="*40)
print(f"### FINAL MLFLOW METRIC TO LOG ###")
print(f"avg_vendor_diversity_top5: {mlflow_metric:.4f}")
print("="*40)


### FINAL MLFLOW METRIC TO LOG ###
avg_vendor_diversity_top5: 2.1500
